# Kaggle submission – Assignment 1 baseline (GBDT residual)
Self-contained version of the repository's `src/` package for the Kaggle code-competition environment (internet off). Add the competition data, then *Save Version → Save & Run All* and submit.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
FILES = {}
FILES['__init__.py'] = '"""Reusable code for the CENG313 NFL Big Data Bowl 2026 project."""\nfrom . import data, models, utils  # noqa: F401\n'
FILES['data.py'] = '"""Data loading, field normalization and feature tables for the\nNFL Big Data Bowl 2026 (Prediction) competition.\n\nConventions\n-----------\n* Field: x in [0, 120] yards (including end zones), y in [0, 53.3] yards.\n* Tracking is sampled at 10 Hz, so one frame = 0.1 s.\n* NGS angles: 0 deg points to +y and angles grow clockwise, so a player\n  moving with speed ``s`` and direction ``dir`` has\n  ``vx = s * sin(dir)`` and ``vy = s * cos(dir)``.\n* All modelling happens in a *normalized* frame where the offense always\n  moves towards increasing x (plays with ``play_direction == "left"`` are\n  rotated by 180 degrees). Predictions are rotated back at the end.\n"""\nfrom __future__ import annotations\n\nimport glob\nimport os\nfrom typing import Iterable, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nFIELD_X = 120.0\nFIELD_Y = 53.3\nFPS = 10.0\nDT = 1.0 / FPS\n\nPLAY_KEYS = ["game_id", "play_id"]\nPLAYER_KEYS = PLAY_KEYS + ["nfl_id"]\nFRAME_KEYS = PLAYER_KEYS + ["frame_id"]\n\nCOMPETITION = "nfl-big-data-bowl-2026-prediction"\n\nROLE_MAP = {"Targeted Receiver": 0, "Defensive Coverage": 1, "Other Route Runner": 2, "Passer": 3}\nPOSITIONS = ["QB", "RB", "FB", "WR", "TE", "CB", "S", "FS", "SS", "ILB", "OLB", "MLB", "LB", "DE", "DT", "NT"]\nPOS_MAP = {p: i for i, p in enumerate(POSITIONS)}\n\n_INPUT_DTYPES = {\n    "game_id": "int64", "play_id": "int32", "nfl_id": "int32", "frame_id": "int16",\n    "absolute_yardline_number": "int16", "player_weight": "int16", "num_frames_output": "int16",\n    "x": "float32", "y": "float32", "s": "float32", "a": "float32", "dir": "float32", "o": "float32",\n    "ball_land_x": "float32", "ball_land_y": "float32",\n    "play_direction": "category", "player_position": "category", "player_side": "category",\n    "player_role": "category",\n}\n_OUTPUT_DTYPES = {"game_id": "int64", "play_id": "int32", "nfl_id": "int32", "frame_id": "int16",\n                  "x": "float32", "y": "float32"}\n\n\n# --------------------------------------------------------------------------- loading\ndef find_data_dir(candidates: Optional[Iterable[str]] = None) -> str:\n    """Return the first directory that contains the competition\'s ``train/`` folder."""\n    env = os.environ.get("NFL_DATA_DIR")\n    paths = [env] if env else []\n    paths += list(candidates or []) + [\n        f"/kaggle/input/{COMPETITION}",\n        f"/content/{COMPETITION}",\n        f"/content/drive/MyDrive/{COMPETITION}",\n        f"./data/{COMPETITION}",\n        f"./{COMPETITION}",\n        f"../{COMPETITION}",\n    ]\n    for p in paths:\n        if p and os.path.isdir(os.path.join(p, "train")):\n            return os.path.abspath(p)\n    raise FileNotFoundError(\n        "Competition data not found. Set NFL_DATA_DIR or place the data in ./data/" + COMPETITION)\n\n\ndef _week_files(data_dir: str, kind: str, weeks: Optional[Iterable[int]]) -> list:\n    files = sorted(glob.glob(os.path.join(data_dir, "train", f"{kind}_2023_w*.csv")))\n    if weeks is not None:\n        wanted = {f"w{int(w):02d}" for w in weeks}\n        files = [f for f in files if os.path.basename(f).split("_")[-1][:3] in wanted]\n    if not files:\n        raise FileNotFoundError(f"No {kind} files in {data_dir}/train")\n    return files\n\n\ndef load_train(data_dir: str, weeks: Optional[Iterable[int]] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """Load the weekly input (pre-pass) and output (post-pass) files.\n\n    A ``week`` column is added to both frames so results can be sliced by week.\n    """\n    ins, outs = [], []\n    for f in _week_files(data_dir, "input", weeks):\n        df = pd.read_csv(f, dtype=_INPUT_DTYPES)\n        df["week"] = np.int8(os.path.basename(f)[-6:-4])\n        ins.append(df)\n    for f in _week_files(data_dir, "output", weeks):\n        df = pd.read_csv(f, dtype=_OUTPUT_DTYPES)\n        df["week"] = np.int8(os.path.basename(f)[-6:-4])\n        outs.append(df)\n    inp = pd.concat(ins, ignore_index=True)\n    out = pd.concat(outs, ignore_index=True)\n    for c in ["play_direction", "player_position", "player_side", "player_role"]:\n        inp[c] = inp[c].astype("category")\n    return inp, out\n\n\ndef load_test(data_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """Load the public ``test.csv`` (rows to predict) and ``test_input.csv``."""\n    test = pd.read_csv(os.path.join(data_dir, "test.csv"))\n    test_input = pd.read_csv(os.path.join(data_dir, "test_input.csv"), dtype=_INPUT_DTYPES)\n    return test, test_input\n\n\n# --------------------------------------------------------------------------- normalization\ndef normalize_direction(df: pd.DataFrame) -> pd.DataFrame:\n    """Rotate left-moving plays by 180 degrees so the offense always moves right.\n\n    Adds ``is_left`` (1 for rotated plays) and ``los_x`` (line of scrimmage in the\n    normalized frame). ``absolute_yardline_number`` is given in the original x\n    coordinate, so it is rotated together with x.\n    """\n    df = df.copy()\n    left = (df["play_direction"].astype(str) == "left").to_numpy()\n    df["is_left"] = left.astype(np.int8)\n    for c, L in [("x", FIELD_X), ("y", FIELD_Y), ("ball_land_x", FIELD_X), ("ball_land_y", FIELD_Y)]:\n        if c in df:\n            df[c] = np.where(left, L - df[c], df[c]).astype(np.float32)\n    for c in ["dir", "o"]:\n        if c in df:\n            df[c] = np.where(left, (df[c] + 180.0) % 360.0, df[c]).astype(np.float32)\n    if "absolute_yardline_number" in df:\n        df["los_x"] = np.where(left, FIELD_X - df["absolute_yardline_number"],\n                               df["absolute_yardline_number"]).astype(np.float32)\n    return df\n\n\ndef flip_xy(x: np.ndarray, y: np.ndarray, is_left: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    """Apply (or undo - the transform is its own inverse) the 180 degree rotation."""\n    left = np.asarray(is_left).astype(bool)\n    return np.where(left, FIELD_X - x, x), np.where(left, FIELD_Y - y, y)\n\n\ndef height_to_inches(h) -> float:\n    try:\n        feet, inch = str(h).split("-")\n        return int(feet) * 12 + int(inch)\n    except (ValueError, AttributeError):\n        return np.nan\n\n\ndef angle_diff(a, b):\n    """Absolute smallest difference between two angles in degrees, in [0, 180]."""\n    return np.abs((np.asarray(a) - np.asarray(b) + 180.0) % 360.0 - 180.0)\n\n\n# --------------------------------------------------------------------------- feature tables\ndef add_kinematics(df: pd.DataFrame) -> pd.DataFrame:\n    """Velocity / acceleration components from s, a, dir and from position differences."""\n    df = df.sort_values(FRAME_KEYS).copy()\n    rad = np.deg2rad(df["dir"].to_numpy())\n    df["vx"] = (df["s"].to_numpy() * np.sin(rad)).astype(np.float32)\n    df["vy"] = (df["s"].to_numpy() * np.cos(rad)).astype(np.float32)\n    g = df.groupby(PLAYER_KEYS, sort=False, observed=True)\n    df["vx_pos"] = ((df["x"] - g["x"].shift(1)) / DT).astype(np.float32)\n    df["vy_pos"] = ((df["y"] - g["y"].shift(1)) / DT).astype(np.float32)\n    # acceleration vector from the change in velocity over the last 2 frames\n    df["ax_est"] = ((df["vx"] - g["vx"].shift(2)) / (2 * DT)).astype(np.float32)\n    df["ay_est"] = ((df["vy"] - g["vy"].shift(2)) / (2 * DT)).astype(np.float32)\n    df["s_lag5"] = g["s"].shift(5)\n    df["dir_lag5"] = g["dir"].shift(5)\n    return df\n\n\ndef build_player_table(inp: pd.DataFrame, only_to_predict: bool = True) -> pd.DataFrame:\n    """One row per player at the last pre-pass frame, with basic features.\n\n    The features are intentionally *simple* (kinematics, ball landing spot,\n    player attributes). Interaction features between players are left for the\n    proposed solution in Assignment 2.\n    """\n    df = add_kinematics(normalize_direction(inp))\n    last = df.groupby(PLAYER_KEYS, sort=False, observed=True).tail(1).copy()\n    last = last.rename(columns={"frame_id": "last_input_frame"})\n\n    last["s_change5"] = last["s"] - last["s_lag5"]\n    last["dir_change5"] = (last["dir"] - last["dir_lag5"] + 180.0) % 360.0 - 180.0\n    for c in ["vx_pos", "vy_pos", "ax_est", "ay_est", "s_change5", "dir_change5"]:\n        last[c] = last[c].fillna(0.0)\n\n    T = last["num_frames_output"].to_numpy() * DT\n    last["flight_time"] = T\n    last["dx_ball"] = last["ball_land_x"] - last["x"]\n    last["dy_ball"] = last["ball_land_y"] - last["y"]\n    last["dist_ball"] = np.hypot(last["dx_ball"], last["dy_ball"])\n    last["angle_to_ball"] = np.rad2deg(np.arctan2(last["dx_ball"], last["dy_ball"])) % 360.0\n    last["dir_vs_ball"] = angle_diff(last["dir"], last["angle_to_ball"])\n    last["o_vs_ball"] = angle_diff(last["o"], last["angle_to_ball"])\n    last["req_speed"] = last["dist_ball"] / np.maximum(T, DT)\n    last["closing_speed"] = (last["vx"] * last["dx_ball"] + last["vy"] * last["dy_ball"]) / (last["dist_ball"] + 1e-6)\n\n    last["height_in"] = last["player_height"].map(height_to_inches)\n    birth = pd.to_datetime(last["player_birth_date"], errors="coerce")\n    last["age"] = (pd.Timestamp("2023-09-07") - birth).dt.days / 365.25\n    last["role_id"] = last["player_role"].astype(str).map(ROLE_MAP).fillna(-1).astype(np.int8)\n    last["pos_id"] = last["player_position"].astype(str).map(POS_MAP).fillna(-1).astype(np.int8)\n    last["is_offense"] = (last["player_side"].astype(str) == "Offense").astype(np.int8)\n    last["x_from_los"] = last["x"] - last["los_x"]\n    last["dist_sideline"] = np.minimum(last["y"], FIELD_Y - last["y"])\n\n    if only_to_predict:\n        last = last[last["player_to_predict"].astype(bool)]\n    return last.reset_index(drop=True)\n\n\nPLAYER_FEATURES = [\n    "x", "y", "s", "a", "dir", "o", "vx", "vy", "vx_pos", "vy_pos", "ax_est", "ay_est",\n    "s_change5", "dir_change5", "flight_time", "num_frames_output", "ball_land_x", "ball_land_y",\n    "dx_ball", "dy_ball", "dist_ball", "angle_to_ball", "dir_vs_ball", "o_vs_ball", "req_speed",\n    "closing_speed", "height_in", "player_weight", "age", "role_id", "pos_id", "is_offense",\n    "x_from_los", "dist_sideline",\n]\nFRAME_FEATURES = ["frame_id", "t", "t_frac", "frames_left", "cv_dx", "cv_dy", "ball_dx_lin", "ball_dy_lin"]\nFEATURES = PLAYER_FEATURES + FRAME_FEATURES\n\n\ndef expand_to_frames(players: pd.DataFrame, frames: Optional[pd.DataFrame] = None) -> pd.DataFrame:\n    """Create one row per (player, output frame).\n\n    If ``frames`` (with ``FRAME_KEYS``) is given, exactly those rows are produced\n    (in that order); otherwise frames ``1..num_frames_output`` are generated.\n    """\n    if frames is None:\n        reps = players["num_frames_output"].astype(int).to_numpy()\n        rows = players.loc[players.index.repeat(reps)].reset_index(drop=True)\n        rows["frame_id"] = np.concatenate([np.arange(1, r + 1) for r in reps]).astype(np.int16)\n    else:\n        rows = frames[FRAME_KEYS].merge(players, on=PLAYER_KEYS, how="left")\n    t = rows["frame_id"].to_numpy() * DT\n    rows["t"] = t\n    rows["t_frac"] = rows["frame_id"] / rows["num_frames_output"]\n    rows["frames_left"] = rows["num_frames_output"] - rows["frame_id"]\n    rows["cv_dx"] = rows["vx"] * t\n    rows["cv_dy"] = rows["vy"] * t\n    rows["ball_dx_lin"] = rows["dx_ball"] * np.minimum(rows["t_frac"], 1.0)\n    rows["ball_dy_lin"] = rows["dy_ball"] * np.minimum(rows["t_frac"], 1.0)\n    return rows\n\n\ndef build_training_frame(inp: pd.DataFrame, out: pd.DataFrame) -> pd.DataFrame:\n    """Join the player table with the post-pass targets (in the normalized frame)."""\n    players = build_player_table(inp)\n    rows = expand_to_frames(players, frames=out)\n    tx, ty = flip_xy(out["x"].to_numpy(), out["y"].to_numpy(), rows["is_left"].to_numpy())\n    rows["x_true"] = tx.astype(np.float32)\n    rows["y_true"] = ty.astype(np.float32)\n    return rows\n'
FILES['models.py'] = '"""Baseline models for post-pass player trajectory prediction.\n\nEvery model follows the same small interface::\n\n    model.fit(train_rows)            # rows from data.expand_to_frames / build_training_frame\n    x_pred, y_pred = model.predict(rows)\n\nPredictions are made in the normalized frame (offense moving right) and are\nclipped to the field. Physics baselines need no training; the fitted\nbaselines learn a handful of parameters (or a residual model) from data.\n"""\nfrom __future__ import annotations\n\nfrom typing import Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.linear_model import Ridge\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom .data import FEATURES, FIELD_X, FIELD_Y\n\nPred = Tuple[np.ndarray, np.ndarray]\n\n\ndef _clip(x, y) -> Pred:\n    return np.clip(np.asarray(x, dtype=np.float64), 0, FIELD_X), np.clip(np.asarray(y, dtype=np.float64), 0, FIELD_Y)\n\n\nclass BaselineModel:\n    name = "base"\n    trainable = False\n\n    def fit(self, rows: pd.DataFrame) -> "BaselineModel":\n        return self\n\n    def predict(self, rows: pd.DataFrame) -> Pred:  # pragma: no cover - abstract\n        raise NotImplementedError\n\n    def __repr__(self):\n        return f"{self.__class__.__name__}()"\n\n\n# --------------------------------------------------------------------------- physics baselines\nclass LastPosition(BaselineModel):\n    """B0: the player freezes at the position of the last pre-pass frame."""\n    name = "Last position"\n\n    def predict(self, rows):\n        return _clip(rows["x"], rows["y"])\n\n\nclass ConstantVelocity(BaselineModel):\n    """B1: the player keeps the speed and direction measured at the pass."""\n    name = "Constant velocity"\n\n    def predict(self, rows):\n        return _clip(rows["x"] + rows["cv_dx"], rows["y"] + rows["cv_dy"])\n\n\nclass ConstantAcceleration(BaselineModel):\n    """B2: constant acceleration, using the acceleration vector estimated from\n    the change in velocity over the last frames (capped at ``max_speed``)."""\n    name = "Constant acceleration"\n\n    def __init__(self, max_speed: float = 11.0):\n        self.max_speed = max_speed\n\n    def predict(self, rows):\n        t = rows["t"].to_numpy()\n        vx0, vy0 = rows["vx"].to_numpy(), rows["vy"].to_numpy()\n        ax, ay = rows["ax_est"].to_numpy(), rows["ay_est"].to_numpy()\n        dx = vx0 * t + 0.5 * ax * t ** 2\n        dy = vy0 * t + 0.5 * ay * t ** 2\n        # do not allow the implied average speed to exceed a human limit\n        avg = np.hypot(dx, dy) / np.maximum(t, 1e-6)\n        scale = np.minimum(1.0, self.max_speed / np.maximum(avg, 1e-6))\n        return _clip(rows["x"] + dx * scale, rows["y"] + dy * scale)\n\n\nclass DampedVelocity(BaselineModel):\n    """B3: velocity decays exponentially, v(t) = v0 * exp(-k t).\n\n    Displacement = v0 * (1 - exp(-k t)) / k. The single parameter ``k`` is\n    fitted on the training rows by grid search.\n    """\n    name = "Damped velocity"\n    trainable = True\n\n    def __init__(self, k: Optional[float] = None, grid=np.linspace(0.0, 1.5, 61)):\n        self.k = k\n        self.grid = grid\n\n    @staticmethod\n    def _factor(t, k):\n        return t if k < 1e-9 else (1 - np.exp(-k * t)) / k\n\n    def fit(self, rows):\n        t = rows["t"].to_numpy()\n        ex = (rows["x_true"] - rows["x"]).to_numpy()\n        ey = (rows["y_true"] - rows["y"]).to_numpy()\n        vx, vy = rows["vx"].to_numpy(), rows["vy"].to_numpy()\n        best = min(self.grid, key=lambda k: np.mean((ex - vx * self._factor(t, k)) ** 2 + (ey - vy * self._factor(t, k)) ** 2))\n        self.k = float(best)\n        return self\n\n    def predict(self, rows):\n        f = self._factor(rows["t"].to_numpy(), self.k)\n        return _clip(rows["x"] + rows["vx"] * f, rows["y"] + rows["vy"] * f)\n\n    def __repr__(self):\n        return f"DampedVelocity(k={self.k})"\n\n\nclass StraightToBall(BaselineModel):\n    """B4: every player runs in a straight line and reaches the ball landing\n    spot exactly when the ball arrives."""\n    name = "Straight to ball"\n\n    def predict(self, rows):\n        return _clip(rows["x"] + rows["ball_dx_lin"], rows["y"] + rows["ball_dy_lin"])\n\n\nclass RolePhysicsBlend(BaselineModel):\n    """B5: per-role blend of damped velocity and straight-to-ball motion.\n\n    For each player role r and flight-progress bin b a weight w[r, b] is\n    fitted by least squares:  disp = (1 - w) * damped + w * to_ball.\n    Captures, e.g., that targeted receivers bend towards the ball while other\n    route runners mostly keep running.\n    """\n    name = "Role-aware physics blend"\n    trainable = True\n\n    def __init__(self, n_bins: int = 5):\n        self.n_bins = n_bins\n        self.damped = DampedVelocity()\n        self.w = {}\n\n    def _parts(self, rows):\n        f = DampedVelocity._factor(rows["t"].to_numpy(), self.damped.k)\n        dvx, dvy = rows["vx"].to_numpy() * f, rows["vy"].to_numpy() * f\n        bx, by = rows["ball_dx_lin"].to_numpy(), rows["ball_dy_lin"].to_numpy()\n        bins = np.minimum((rows["t_frac"].to_numpy() * self.n_bins).astype(int), self.n_bins - 1)\n        return dvx, dvy, bx, by, bins\n\n    def fit(self, rows):\n        self.damped.fit(rows)\n        dvx, dvy, bx, by, bins = self._parts(rows)\n        ex = rows["x_true"].to_numpy() - rows["x"].to_numpy()\n        ey = rows["y_true"].to_numpy() - rows["y"].to_numpy()\n        role = rows["role_id"].to_numpy()\n        for r in np.unique(role):\n            for b in range(self.n_bins):\n                m = (role == r) & (bins == b)\n                if m.sum() < 50:\n                    self.w[(r, b)] = 0.0\n                    continue\n                # minimise |e - d - w (bl - d)|^2  ->  closed form\n                ux, uy = bx[m] - dvx[m], by[m] - dvy[m]\n                rx, ry = ex[m] - dvx[m], ey[m] - dvy[m]\n                w = (ux @ rx + uy @ ry) / max(ux @ ux + uy @ uy, 1e-9)\n                self.w[(r, b)] = float(np.clip(w, 0.0, 1.0))\n        return self\n\n    def predict(self, rows):\n        dvx, dvy, bx, by, bins = self._parts(rows)\n        role = rows["role_id"].to_numpy()\n        w = np.array([self.w.get((r, b), 0.0) for r, b in zip(role, bins)])\n        return _clip(rows["x"] + (1 - w) * dvx + w * bx, rows["y"] + (1 - w) * dvy + w * by)\n\n\n# --------------------------------------------------------------------------- learned residual baselines\nclass _ResidualModel(BaselineModel):\n    """Learns the correction on top of constant velocity:\n    x = x0 + vx t + f_x(features),  y = y0 + vy t + f_y(features)."""\n    trainable = True\n    features = FEATURES\n\n    def _make(self):  # pragma: no cover - abstract\n        raise NotImplementedError\n\n    def _X(self, rows):\n        return rows[self.features].astype(np.float32).fillna(0.0).to_numpy()\n\n    def fit(self, rows):\n        X = self._X(rows)\n        rx = (rows["x_true"] - rows["x"] - rows["cv_dx"]).to_numpy()\n        ry = (rows["y_true"] - rows["y"] - rows["cv_dy"]).to_numpy()\n        self.mx, self.my = self._make().fit(X, rx), self._make().fit(X, ry)\n        return self\n\n    def predict(self, rows):\n        X = self._X(rows)\n        return _clip(rows["x"] + rows["cv_dx"] + self.mx.predict(X), rows["y"] + rows["cv_dy"] + self.my.predict(X))\n\n\nclass RidgeResidual(_ResidualModel):\n    """B6: linear (ridge) regression on the residual. Each feature is also\n    multiplied by t so the correction can grow with time."""\n    name = "Ridge residual"\n\n    def __init__(self, alpha: float = 10.0):\n        self.alpha = alpha\n\n    def _X(self, rows):\n        X = super()._X(rows)\n        t = rows["t"].to_numpy(np.float32)[:, None]\n        return np.hstack([X, X * t])\n\n    def _make(self):\n        return make_pipeline(StandardScaler(), Ridge(alpha=self.alpha))\n\n\nclass GBDTResidual(_ResidualModel):\n    """B7: histogram gradient boosting on the residual (scikit-learn)."""\n    name = "GBDT residual"\n\n    def __init__(self, max_iter: int = 300, learning_rate: float = 0.1, max_leaf_nodes: int = 63,\n                 random_state: int = 42):\n        self.params = dict(max_iter=max_iter, learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes,\n                           min_samples_leaf=40, l2_regularization=1.0, random_state=random_state,\n                           early_stopping=False)\n\n    def _make(self):\n        return HistGradientBoostingRegressor(**self.params)\n\n\nALL_BASELINES = [LastPosition, ConstantVelocity, ConstantAcceleration, DampedVelocity, StraightToBall,\n                 RolePhysicsBlend, RidgeResidual, GBDTResidual]\n'
FILES['utils.py'] = '"""Metric, cross-validation and plotting helpers."""\nfrom __future__ import annotations\n\nimport time\nfrom contextlib import contextmanager\nfrom typing import Callable, Dict, Optional\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import GroupKFold\n\nfrom .data import FIELD_X, FIELD_Y, PLAY_KEYS\n\n\n# --------------------------------------------------------------------------- metric\ndef rmse(x_true, y_true, x_pred, y_pred) -> float:\n    """Competition metric: sqrt( sum[(dx)^2 + (dy)^2] / (2N) )."""\n    dx = np.asarray(x_true) - np.asarray(x_pred)\n    dy = np.asarray(y_true) - np.asarray(y_pred)\n    return float(np.sqrt(np.mean(dx ** 2 + dy ** 2) / 2.0))\n\n\ndef rmse_by(rows: pd.DataFrame, x_pred, y_pred, by) -> pd.Series:\n    """RMSE broken down by one or more columns of ``rows``."""\n    se = ((rows["x_true"].to_numpy() - x_pred) ** 2 + (rows["y_true"].to_numpy() - y_pred) ** 2) / 2.0\n    return pd.Series(se, index=rows.index).groupby([rows[b] for b in np.atleast_1d(by)], observed=True).mean() ** 0.5\n\n\n# --------------------------------------------------------------------------- validation\ndef group_kfold_oof(make_model: Callable, rows: pd.DataFrame, n_splits: int = 5, group_col: str = "game_id",\n                    verbose: bool = True) -> Dict:\n    """Out-of-fold predictions with folds split by game (no game in train and valid).\n\n    Returns a dict with ``x``/``y`` OOF predictions, per-fold scores, the overall\n    OOF RMSE, the fitted models and the elapsed time.\n    """\n    xo = np.zeros(len(rows))\n    yo = np.zeros(len(rows))\n    folds, models = [], []\n    start = time.time()\n    for k, (tr, va) in enumerate(GroupKFold(n_splits=n_splits).split(rows, groups=rows[group_col])):\n        m = make_model().fit(rows.iloc[tr])\n        px, py = m.predict(rows.iloc[va])\n        xo[va], yo[va] = px, py\n        s = rmse(rows["x_true"].iloc[va], rows["y_true"].iloc[va], px, py)\n        folds.append(s)\n        models.append(m)\n        if verbose:\n            print(f"  fold {k}: {s:.4f}")\n    return {"x": xo, "y": yo, "folds": folds, "models": models,\n            "rmse": rmse(rows["x_true"], rows["y_true"], xo, yo), "seconds": time.time() - start}\n\n\n@contextmanager\ndef timer(label: str):\n    t0 = time.time()\n    yield\n    print(f"[{label}] {time.time() - t0:.1f}s")\n\n\n# --------------------------------------------------------------------------- plotting\ndef draw_field(ax, xlim=(0, FIELD_X), ylim=(0, FIELD_Y), turf="#2d6a4f"):\n    import matplotlib.pyplot as plt\n    ax.add_patch(plt.Rectangle((0, 0), FIELD_X, FIELD_Y, color=turf, zorder=0))\n    for x in (0, 110):\n        ax.add_patch(plt.Rectangle((x, 0), 10, FIELD_Y, color="#1b4332", zorder=0))\n    for x in range(10, 111, 5):\n        ax.plot([x, x], [0, FIELD_Y], color="white", lw=1.0 if x % 10 == 0 else 0.4, alpha=0.6, zorder=1)\n    ax.set_xlim(*xlim)\n    ax.set_ylim(*ylim)\n    ax.set_aspect("equal", adjustable="datalim")\n    ax.set_xticks([])\n    ax.set_yticks([])\n    return ax\n\n\ndef plot_play(inp: pd.DataFrame, out: pd.DataFrame, game_id: int, play_id: int, preds: Optional[pd.DataFrame] = None,\n              ax=None, zoom: bool = True, title: Optional[str] = None):\n    """Plot one play in original coordinates.\n\n    Dotted: pre-pass path (input). Solid: true post-pass path (output).\n    Dashed cyan: predicted post-pass path (``preds`` with nfl_id, x, y). Star: ball landing.\n    """\n    import matplotlib.pyplot as plt\n    pi = inp[(inp.game_id == game_id) & (inp.play_id == play_id)]\n    po = out[(out.game_id == game_id) & (out.play_id == play_id)]\n    if ax is None:\n        _, ax = plt.subplots(figsize=(12, 5.5))\n    draw_field(ax)\n    colors = {"Offense": "#ffd166", "Defense": "#a8dadc"}\n    for nid, g in pi.groupby("nfl_id"):\n        side = str(g.player_side.iloc[0])\n        role = str(g.player_role.iloc[0])\n        ax.plot(g.x, g.y, ":", color=colors.get(side, "white"), lw=1.5)\n        o = po[po.nfl_id == nid]\n        if len(o):\n            ax.plot(o.x, o.y, "-", color="#e76f51" if role == "Targeted Receiver" else colors.get(side, "white"), lw=2.5)\n        if preds is not None:\n            p = preds[preds.nfl_id == nid]\n            if len(p):\n                ax.plot(p.x, p.y, "--", color="#00e5ff", lw=2)\n    bx, by = pi.ball_land_x.iloc[0], pi.ball_land_y.iloc[0]\n    ax.plot(bx, by, "*", ms=20, color="#e76f51", mec="white", zorder=5)\n    if zoom:\n        xs = pd.concat([pi.x, po.x, pd.Series([bx])])\n        ys = pd.concat([pi.y, po.y, pd.Series([by])])\n        ax.set_xlim(max(-1, xs.min() - 5), min(121, xs.max() + 5))\n        ax.set_ylim(max(-1, ys.min() - 4), min(54.3, ys.max() + 4))\n    ax.set_title(title or f"game {game_id} · play {play_id} · {pi.play_direction.iloc[0]} · "\n                          f"{int(pi.num_frames_output.iloc[0])} frames in the air")\n    return ax\n'
for n, c in FILES.items():
    open(os.path.join('src', n), 'w').write(c)
print('src/ written')

In [ ]:
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
from src import data as D, models as M

DATA_DIR = '/kaggle/input/nfl-big-data-bowl-2026-prediction'
inp, out = D.load_train(DATA_DIR)
rows = D.build_training_frame(inp, out)
model = M.GBDTResidual().fit(rows)
del inp, out
print('trained on', len(rows), 'rows')

In [ ]:
import polars as pl

def predict(test, test_input):
    t = test.to_pandas() if hasattr(test, 'to_pandas') else test
    ti = test_input.to_pandas() if hasattr(test_input, 'to_pandas') else test_input
    for c in ['play_direction', 'player_position', 'player_side', 'player_role']:
        ti[c] = ti[c].astype('category')
    rows_t = D.expand_to_frames(D.build_player_table(ti), frames=t)
    x, y = model.predict(rows_t)
    x, y = D.flip_xy(x, y, rows_t['is_left'].to_numpy())
    assert len(x) == len(t)
    return pl.DataFrame({'x': x, 'y': y})

import kaggle_evaluation.nfl_inference_server
server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    server.serve()
else:
    server.run_local_gateway((DATA_DIR,))